In [ ]:
##import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## Data Collection
df = pd.read_csv("Housing.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
## missing values check
df.isnull().sum()

In [ ]:
## checking for duplicate records
df.duplicated()

In [ ]:
## checking for duplicate records
df[df.duplicated()]  ## no duplicate records

In [ ]:
df.head()

In [ ]:
df["furnishingstatus"].unique()

## Feature Engineering 

### Feature Extraction

In [ ]:
## get all columns
df.columns

In [ ]:
df.info()

In [ ]:
df['area'].dtype

In [ ]:
## Numerical Feature
numerical_feature = [feature for feature in df.columns if df[feature].dtype != 'O']
numerical_feature

In [ ]:
## Categorical Feature 
categorical_feature = [feature for feature in df.columns if df[feature].dtype == 'O']
categorical_feature

In [ ]:
## Discrete Feature
discrete_feature = [feature for feature in numerical_feature if len(df[feature].unique())<25]
discrete_feature

In [ ]:
df['bathrooms'].unique() ## < 25

In [ ]:
df['area'].unique() ## < 25

In [ ]:
## Continuous Feature
continuous_feature = [feature for feature in numerical_feature if feature not in discrete_feature]
continuous_feature

In [ ]:
print(f"Numerical Features: {numerical_feature}")
print(f"Length of numerical features: {len(numerical_feature)}")
print(f"Categorical Features: {categorical_feature} ")
print(f"Length of categorical feature: {len(categorical_feature)}")
print(f"Discrete Features: {discrete_feature}")
print(f"Length of discrete feature: {len(discrete_feature)}")
print(f"Continuous Features: {continuous_feature}")
print(f"Length of continuous featue: {len(continuous_feature)}")


### Independent Feature and Dependent Feature

In [ ]:
## independent feature
X = df.drop(["price"],axis=1)
X.head()

In [ ]:
## independent feature
y = df["price"]
y

## Column Transformer for transforming values

In [ ]:

## Create Column Transformer with 3 types of transformer i.e. StandardScaler() , OneHotEncoder(), ColumnTransformer()
### numerical feature from independent features
num_features = X.select_dtypes(exclude = "object").columns
cat_features = X.select_dtypes(include = "object").columns

## transformer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

oh_encoder = OneHotEncoder(drop='first')
std_scaler = StandardScaler()

preprocessor = ColumnTransformer(
    [
        ("One Hot Encoder",oh_encoder,cat_features),
        ("StandardScaler",std_scaler,num_features)
    ]
)

In [ ]:
preprocessor

In [ ]:
X

In [ ]:
y

In [ ]:
## applying the transformation to the training dataset (fit_transform)
X = preprocessor.fit_transform(X)

In [ ]:
temp_df=pd.DataFrame(X)

In [ ]:
temp_df.corr()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(temp_df.corr(),annot=True)


## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.shape,X_test.shape

In [ ]:
y_train.shape,y_test.shape

## Model Training
### Model Evaluation

In [ ]:
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error,recall_score,precision_score

In [ ]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    recall_score = recall_score(true,predicted)
    precision_score = precision_score(true,predicted)
    return mae, rmse, r2_square,recall_score,precision_score

In [ ]:
## create dictionary of models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression":Ridge(),
    "Lasso Regression":Lasso(),
    "Kneighbor Regressor":KNeighborsRegressor(),
    "Decision Tree Regressor":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor()
}

## looping for each model ,predicting the values and model evaluation
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train) ## model training

    ## Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # print(y_train_pred)
    # print(y_test_pred)
    
    ## model evaluation on train dataset
    model_train_mae,model_train_rmse, model_train_r2,recall_score,precision_score= evaluate_model(y_train,y_train_pred)

    ## model evaluation on test dataset
    model_test_mae,model_test_rmse, model_test_r2,recall_score,precision_score= evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))
    

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

## Hyper parameter tunning

### RandomizedCV

In [ ]:
#Initialize few parameter for Hyperparamter tuning
dt_params = {
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15, 20, 25, 30], # 'None' means unlimited depth
    'min_samples_split': [2, 5, 10, 20],
    # 'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2', 0.8, 0.5], # 0.8 means 80% of features
    # 'max_leaf_nodes': [None, 10, 20, 30, 50],
    # 'min_impurity_decrease': [0.0, 0.01, 0.05, 0.1],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.05, 0.1,0.0005]
}

rf_params={
    'n_estimators': [100, 200, 300, 500], # Number of trees in the forest
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
    'max_depth': [None, 10, 20, 30], # None means unlimited depth
    'min_samples_split': [2, 5, 10],
    'max_features': [1.0, 'sqrt', 'log2', 0.8], # For Random Forest, 'sqrt' (or 0.33) is often a good default
    # 'bootstrap': [True, False], # Whether bootstrap samples are used when building trees
    # 'oob_score': [False], # Usually set to True if you want to use OOB samples to estimate the generalization accuracy
    # 'ccp_alpha': [0.0, 0.001, 0.01], # Cost-Complexity Pruning
    # 'max_samples': [None, 0.7, 0.9], # If bootstrap=True, this sets the fraction of samples to draw. Not always tuned initially.
    # 'max_leaf_nodes': [None, 20, 50], # Less common to tune heavily if max_depth is tuned.
    # 'min_impurity_decrease': [0.0, 0.01], # Less common to tune heavily if others are used for regularization.
}

adb_params={
    'n_estimators': [50, 100, 200, 300, 500], # Number of boosting stages (weak learners)
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0], # Shrinks the contribution of each estimator
    'loss': ['linear', 'square', 'exponential'], # Type of loss function to use
    # 'estimator': [ # The base estimator. Often a DecisionTreeRegressor.
    #     # Define base estimators with various depths. Shallow trees are common for AdaBoost.
    #     DecisionTreeRegressor(max_depth=1), # Very common: "stumps"
    #     DecisionTreeRegressor(max_depth=2),
    #     DecisionTreeRegressor(max_depth=3),
    #     # You could also try other simple models here, but Decision Trees are standard.
    # ],
}

In [ ]:
# Models list for Hyperparameter tuning
randomcv_models=[
    ("Decision Tree",DecisionTreeRegressor(),dt_params),
    ("Random Forest",RandomForestRegressor(),rf_params),
    ("Ada Boost",AdaBoostRegressor(),adb_params)
]

In [ ]:
### Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name,model,params in randomcv_models:
    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        n_jobs=-1,
    )
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"-------------------Best Paramters for {model_name}-----------------------------")
    print(model_param[model_name])

## Re-Model Training with tuned parameters
### Re-Evaluation of Model Performance

In [ ]:
## create dictionary of models
models = {
    "Decision Tree Regressor":DecisionTreeRegressor(splitter = 'best', 
                                                    min_samples_split = 10,
                                                    max_features = 0.5,
                                                    max_depth = 25, 
                                                    criterion = 'absolute_error', 
                                                    ccp_alpha = 0.01),
    "Random Forest Regressor":RandomForestRegressor(
                                                    n_estimators = 300,
                                                    min_samples_split = 5, 
                                                    max_features = 'sqrt', 
                                                    max_depth = 20, 
                                                    criterion = 'absolute_error',
                                                   ),
    "AdaBoost Regressor": AdaBoostRegressor(
        n_estimators = 200, loss  = 'exponential', learning_rate = 0.05
    )
}

## looping for each model ,predicting the values and model evaluation
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train) ## model training

    ## Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # print(y_train_pred)
    # print(y_test_pred)
    
    ## model evaluation on train dataset
    model_train_mae,model_train_rmse, model_train_r2 = evaluate_model(y_train,y_train_pred)

    ## model evaluation on test dataset
    model_test_mae,model_test_rmse, model_test_r2 = evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))
    

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

### GridSearchCV

In [ ]:
#Initialize few parameter for Hyperparamter tuning
dt_params = {
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15, 20, 25, 30], # 'None' means unlimited depth
    'min_samples_split': [2, 5, 10, 20],
    # 'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2', 0.8, 0.5], # 0.8 means 80% of features
    # 'max_leaf_nodes': [None, 10, 20, 30, 50],
    # 'min_impurity_decrease': [0.0, 0.01, 0.05, 0.1],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.05, 0.1,0.0005]
}

rf_params={
    'n_estimators': [100, 200, 300, 500], # Number of trees in the forest
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
    'max_depth': [None, 10, 20, 30], # None means unlimited depth
    'min_samples_split': [2, 5, 10],
    'max_features': [1.0, 'sqrt', 'log2', 0.8], # For Random Forest, 'sqrt' (or 0.33) is often a good default
    # 'bootstrap': [True, False], # Whether bootstrap samples are used when building trees
    # 'oob_score': [False], # Usually set to True if you want to use OOB samples to estimate the generalization accuracy
    # 'ccp_alpha': [0.0, 0.001, 0.01], # Cost-Complexity Pruning
    # 'max_samples': [None, 0.7, 0.9], # If bootstrap=True, this sets the fraction of samples to draw. Not always tuned initially.
    # 'max_leaf_nodes': [None, 20, 50], # Less common to tune heavily if max_depth is tuned.
    # 'min_impurity_decrease': [0.0, 0.01], # Less common to tune heavily if others are used for regularization.
}

adb_params={
    'n_estimators': [50, 100, 200, 300, 500], # Number of boosting stages (weak learners)
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0], # Shrinks the contribution of each estimator
    'loss': ['linear', 'square', 'exponential'], # Type of loss function to use
    # 'estimator': [ # The base estimator. Often a DecisionTreeRegressor.
    #     # Define base estimators with various depths. Shallow trees are common for AdaBoost.
    #     DecisionTreeRegressor(max_depth=1), # Very common: "stumps"
    #     DecisionTreeRegressor(max_depth=2),
    #     DecisionTreeRegressor(max_depth=3),
    #     # You could also try other simple models here, but Decision Trees are standard.
    # ],
}

In [ ]:
# Models list for Hyperparameter tuning
gridcv_models=[
    ("Decision Tree",DecisionTreeRegressor(),dt_params),
    ("Random Forest",RandomForestRegressor(),rf_params),
    ("Ada Boost",AdaBoostRegressor(),adb_params)
]

In [ ]:

#types of cross validation used [stratifiedKFold]
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold()

### Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

model_param = {}
for name,model,params in gridcv_models:
    grid = GridSearchCV(estimator=model, param_grid=params,scoring='accuracy', n_jobs=-1,cv=cv)

    print('------------------------------------')
    print(grid)
    print('------------------------------------')
    grid.fit(X_train,y_train)
    model_param[name] = grid.best_params_

for model_name in model_param:
    print(f"-------------------Best Paramters for {model_name}-----------------------------")
    print(model_param[model_name])

## Re-Model Training with tuned parameters
### Re-Evaluation of Model Performance